# Feature Engineering — YummyAnime Catalog

Input: `data/raw/anime_catalog.csv`  
Output: `data/processed/anime_features.csv`

### New features:
| # | Feature | Description |
|---|---------|-------------|
| 1 | `rating_tier` | Quality tier by site rating (S/A/B/C/D) |
| 2 | `popularity_score` | Normalized popularity (log views * log votes, 0-100) |
| 3 | `engagement_rate` | Views per vote |
| 4 | `genre_count` | Number of genres |
| 5 | `is_multi_genre` | Flag: 3+ genres |
| 6 | `primary_genre` | Primary genre (first in list) |
| 7 | `genre_vector` | One-hot encoding of all unique genres |
| 8 | `decade` | Release decade (1980s / 1990s / 2000s ...) |
| 9 | `era` | Anime era (Classic / Golden Age / Modern / Contemporary / Recent) |
| 10 | `title_age` | Number of years since release |
| 11 | `is_recent` | Flag: released in last 3 years |
| 12 | `is_completed` | Flag: status is completed |
| 13 | `is_ongoing` | Flag: status is ongoing |
| 14 | `is_movie` | Flag: media type is a feature film |
| 15 | `type_simplified` | Simplified type (TV / Movie / OVA / ONA / Special / Other) |
| 16 | `age_rating_ordinal` | Age rating as ordered integer (G=0 ... Rx=5) |
| 17 | `is_family_friendly` | Flag: G or PG rating |
| 18 | `is_adaptation` | Flag: source is manga / light novel / game / etc. |
| 19 | `is_original_work` | Flag: original work (not an adaptation) |
| 20 | `has_dubbing` | Flag: has at least one dubbing |
| 21 | `has_unknown_dubbing` | Flag: only dubbing studio listed is Unknown |
| 22 | `dubbing_count` | Number of dubbing studios |
| 23 | `primary_dubbing` | First dubbing studio |
| 24 | `studio_count` | Number of studios |
| 25 | `primary_studio` | First studio name |
| 26 | `studio_tier` | Studio tier (Top / Mid / Indie / Unknown) |
| 27 | `studio_frequency` | How many titles this studio has in the dataset |
| 28 | `is_prolific_studio` | Flag: studio appears more than once |
| 29 | `director_frequency` | How many titles this director has in the dataset |
| 30 | `is_prolific_director` | Flag: director appears more than once |
| 31 | `hidden_gem` | Flag: high rating + very few votes |
| 32 | `log_votes` | log1p of vote count |
| 33 | `log_views` | log1p of view count |
| 34 | `composite_score` | Quality x popularity composite (0-1) |
| 35 | `alt_names_count` | Number of alternative titles |
| 36 | `title_ru_length` | Character length of Russian title |
| 37 | `slug` | URL slug (for joining with other tables) |

## Imports and Load Data

In [1]:
import ast
import re
import pandas as pd
import numpy as np
from pathlib import Path
from collections import Counter

df = pd.read_csv('../../data/processed/anime_catalog_clean.csv').copy()
print(f'Rows loaded: {len(df)}')
print(f'Columns: {list(df.columns)}')
df.head(3)

Rows loaded: 10142
Columns: ['title_ru', 'alt_names', 'anime_id', 'status', 'type', 'year', 'age_rating', 'genres', 'source', 'studio', 'director', 'dubbing', 'site_rating', 'site_votes', 'site_views', 'url']


,title_ru,alt_names,anime_id,status,type,year,age_rating,genres,source,studio,director,dubbing,site_rating,site_votes,site_views,url
0,Моя история продолжается сегодня,"['Kyou, Watashi no Monogatari ga Hashirimasu. ...",10800,вышел,ONA,2023,G,"['Повседневность ', ' Спорт']",Оригинальная идея,['Studio Colorido'],['Синго Ямасита'],['AniLane'],6.74,19,6,https://old.yummyani.me/catalog/item/moya-isto...
1,Детектив Конан: Цель — Когоро! Секретное рассл...,"[""Detective Conan OVA 05: The Target is Kogoro...",8258,вышел,OVA,2005,PG-13,"['Сёнэн ', ' Детектив ', ' Комедия ', ' Приклю...",Манга,['TMS Entertainment'],['Сато Масато'],['Persona99'],6.65,23,6,https://old.yummyani.me/catalog/item/detektiv-...
2,Сердце дракона: Хроника исследования мира духов,['Dragon Heart: Adventures Beyond This World '...,18742,вышел,Полнометражный фильм,2025,G,"['Фэнтези ', ' Драконы ', ' Сверхъестественное']",Оригинальная идея,['HS Pictures Studio'],['Имакакэ Исаму'],['Unknown'],5.60,5,6,https://old.yummyani.me/catalog/item/dragon-he...


## Preparation: Parse List-valued Columns and Numeric Fields

In [2]:
def parse_list_field(value):
    """Converts a string like "['Studio A', 'Studio B']" into a Python list."""
    if pd.isna(value):
        return []
    try:
        result = ast.literal_eval(str(value))
        if isinstance(result, list):
            return [str(x).strip() for x in result]
        return [str(result).strip()]
    except (ValueError, SyntaxError):
        return []

df['alt_names_list'] = df['alt_names'].apply(parse_list_field)
df['genres_list']    = df['genres'].apply(parse_list_field)
df['studio_list']    = df['studio'].apply(parse_list_field)
df['director_list']  = df['director'].apply(parse_list_field)
df['dubbing_list']   = df['dubbing'].apply(parse_list_field)

df['site_rating_num'] = pd.to_numeric(df['site_rating'], errors='coerce')
df['site_votes_num']  = pd.to_numeric(df['site_votes'],  errors='coerce').astype('Int64')
df['site_views_num']  = pd.to_numeric(df['site_views'],  errors='coerce').astype('Int64')
df['year_num']        = pd.to_numeric(df['year'],         errors='coerce').astype('Int64')

print('Preparation done.')
df[['site_rating_num', 'site_votes_num', 'site_views_num', 'year_num']].describe()

Preparation done.


,site_rating_num,site_votes_num,site_views_num,year_num
count,10142.000000,10142.0,10142.0,10142.0
mean,6.414001,261.780319,50.399132,2012.31128
std,1.830632,686.914423,109.431901,11.948664
min,0.000000,0.0,1.0,1907.0
25%,5.840000,14.0,6.0,2007.0
50%,6.760000,37.0,16.0,2015.0
75%,7.520000,165.0,39.0,2021.0
max,9.540000,9435.0,994.0,2028.0


## Features 1–3: Rating and Popularity

In [3]:
# Feature 1: rating_tier
def rating_tier(r):
    if pd.isna(r) or r == 0: return 'Unrated'
    if r >= 8.5:              return 'S'
    if r >= 7.5:              return 'A'
    if r >= 6.5:              return 'B'
    if r >= 5.0:              return 'C'
    return 'D'

df['rating_tier'] = df['site_rating_num'].apply(rating_tier)
print('rating_tier:', df['rating_tier'].value_counts().to_dict())

rating_tier: {'B': 3287, 'C': 3146, 'A': 2202, 'D': 577, 'Unrated': 510, 'S': 420}


In [4]:
# Feature 2: popularity_score
views = df['site_views_num'].fillna(0).clip(lower=0)
votes = df['site_votes_num'].fillna(0).clip(lower=0)

raw_pop = np.log1p(views) * np.log1p(votes)
pop_max = raw_pop.max()
df['popularity_score'] = (raw_pop / pop_max * 100).round(2) if pop_max > 0 else 0.0

print('popularity_score top-5:')
print(df[['title_ru', 'popularity_score']].sort_values('popularity_score', ascending=False).head())

popularity_score top-5:
                            title_ru  popularity_score
7496                   Атака титанов             100.0
9432                   Звёздное дитя             97.25
6424  Эта фарфоровая кукла влюбилась             96.81
662                          Хоримия             96.27
8402                      Повелитель             96.08


In [5]:
# Feature 3: engagement_rate
views_safe = df['site_views_num'].fillna(0).replace(0, np.nan)
df['engagement_rate'] = (df['site_votes_num'].fillna(0) / views_safe).round(5)
print('engagement_rate — median:', df['engagement_rate'].median())

engagement_rate — median: 3.0


## Features 4–7: Genre Features

In [6]:
# Feature 4: genre_count
df['genre_count'] = df['genres_list'].apply(len)

# Feature 5: is_multi_genre
df['is_multi_genre'] = (df['genre_count'] >= 3).astype(int)

# Feature 6: primary_genre
df['primary_genre'] = df['genres_list'].apply(lambda g: g[0].strip() if g else 'Unknown')

print('genre_count distribution:')
print(df['genre_count'].value_counts().sort_index().head(10))
print('\nprimary_genre top-10:')
print(df['primary_genre'].value_counts().head(10))

genre_count distribution:
genre_count
1      550
2     1120
3     1854
4     2071
5     1817
6     1214
7      785
8      354
9      192
10      68
Name: count, dtype: int64

primary_genre top-10:
primary_genre
Сёнэн          2331
Сэйнэн         1387
Комедия        1355
Драма          1009
Приключения     943
Сёдзё           413
Детектив        368
Фэнтези         366
Этти            313
Фантастика      216
Name: count, dtype: int64


In [7]:
# Feature 7: genre_vector (one-hot — all unique genres)
all_genres_flat = [g.strip() for gl in df['genres_list'] for g in gl if g.strip()]
top_genres = [g for g, _ in Counter(all_genres_flat).most_common()]

for genre in top_genres:
    col = 'genre_' + re.sub(r'[^\u0400-\u04ffa-z0-9]', '_', genre.lower()).strip('_')
    df[col] = df['genres_list'].apply(lambda gl: int(genre in [g.strip() for g in gl]))

genre_cols = [c for c in df.columns if c.startswith('genre_')]
print(f'Created {len(genre_cols)} one-hot genre columns: {genre_cols}')

Created 107 one-hot genre columns: ['genre_count', 'genre_комедия', 'genre_экшен', 'genre_фэнтези', 'genre_приключения', 'genre_драма', 'genre_сёнэн', 'genre_фантастика', 'genre_романтика', 'genre_школьная_жизнь', 'genre_сэйнэн', 'genre_повседневность', 'genre_сверхъестественное', 'genre_не_японское', 'genre_исторический', 'genre_детектив', 'genre_меха', 'genre_этти', 'genre_боевые_искусства', 'genre_суперспособности', 'genre_военная_тематика', 'genre_сёдзё', 'genre_китайское_3d', 'genre_спорт', 'genre_музыка', 'genre_гарем', 'genre_демоны', 'genre_пародия', 'genre_исэкай', 'genre_космос', 'genre_ужасы', 'genre_психология', 'genre_триллер', 'genre_магия', 'genre_игры', 'genre_полицейские', 'genre_мистика', 'genre_самураи', 'genre_вампиры', 'genre_махо_сёдзё', 'genre_дзёсэй', 'genre_бисёнэн', 'genre_сёнэн_ай', 'genre_полулюди', 'genre_киберпанк', 'genre_сёдзё_ай', 'genre_безумие', 'genre_путешествия_во_времени', 'genre_гарем__для_девочек', 'genre_гендерная_интрига', 'genre_любовный_треу

C:\Users\AQTAU\AppData\Local\Temp\ipykernel_18980\804812074.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col] = df['genres_list'].apply(lambda gl: int(genre in [g.strip() for g in gl]))
C:\Users\AQTAU\AppData\Local\Temp\ipykernel_18980\804812074.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col] = df['genres_list'].apply(lambda gl: int(genre in [g.strip() for g in gl]))
C:\Users\AQTAU\AppData\Local\Temp\ipykernel_18980\804812074.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the res

## Features 8–11: Time-based Features

In [8]:
CURRENT_YEAR = pd.Timestamp.now().year

# Feature 8: decade
def to_decade(y):
    if pd.isna(y): return 'Unknown'
    return f'{(int(y) // 10) * 10}s'

df['decade'] = df['year_num'].apply(to_decade)

# Feature 9: era
def to_era(y):
    if pd.isna(y): return 'Unknown'
    y = int(y)
    if y < 1990: return 'Classic'
    if y < 2000: return 'Golden Age'
    if y < 2010: return 'Modern'
    if y < 2020: return 'Contemporary'
    return 'Recent'

df['era'] = df['year_num'].apply(to_era)

# Feature 10: title_age
df['title_age'] = CURRENT_YEAR - df['year_num']

# Feature 11: is_recent
df['is_recent'] = (df['year_num'] >= CURRENT_YEAR - 3).astype(int)

print('decade:', df['decade'].value_counts().sort_index().to_dict())
print('era:',    df['era'].value_counts().to_dict())

C:\Users\AQTAU\AppData\Local\Temp\ipykernel_18980\1014058905.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['decade'] = df['year_num'].apply(to_decade)


decade: {'1900s': 1, '1910s': 2, '1920s': 4, '1930s': 5, '1940s': 2, '1950s': 7, '1960s': 30, '1970s': 88, '1980s': 451, '1990s': 854, '2000s': 1769, '2010s': 3612, '2020s': 3317}
era: {'Contemporary': 3612, 'Recent': 3317, 'Modern': 1769, 'Golden Age': 854, 'Classic': 590}


C:\Users\AQTAU\AppData\Local\Temp\ipykernel_18980\1014058905.py:20: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['era'] = df['year_num'].apply(to_era)
C:\Users\AQTAU\AppData\Local\Temp\ipykernel_18980\1014058905.py:23: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['title_age'] = CURRENT_YEAR - df['year_num']
C:\Users\AQTAU\AppData\Local\Temp\ipykernel_18980\1014058905.py:26: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Co

## Features 12–17: Status and Type

In [9]:
status_lower = df['status'].fillna('').str.lower()

# Feature 12: is_completed
df['is_completed'] = status_lower.str.contains(r'вышел|completed|завершён', regex=True).astype(int)

# Feature 13: is_ongoing
df['is_ongoing'] = status_lower.str.contains(r'онго|ongoing|выходит|выпускается', regex=True).astype(int)

# Feature 14: is_movie
df['is_movie'] = df['type'].fillna('').str.lower().str.contains(r'фильм|movie|полнометр', regex=True).astype(int)

# Feature 15: type_simplified
def simplify_type(t):
    t = str(t).lower()
    if re.search(r'фильм|movie|полнометр', t): return 'Movie'
    if re.search(r'ova|ова|oav',           t): return 'OVA'
    if re.search(r'ona|она|web',           t): return 'ONA'
    if re.search(r'сериал|tv|тв',          t): return 'TV'
    if re.search(r'special|спец',          t): return 'Special'
    return 'Other'

df['type_simplified'] = df['type'].apply(simplify_type)

# Feature 16: age_rating_ordinal
AGE_ORDER = {'G': 0, 'PG': 1, 'PG-13': 2, 'R': 3, 'R+': 4, 'Rx': 5}
df['age_rating_ordinal'] = df['age_rating'].map(AGE_ORDER)

# Feature 17: is_family_friendly
df['is_family_friendly'] = df['age_rating'].isin(['G', 'PG']).astype(int)

print('type_simplified:', df['type_simplified'].value_counts().to_dict())
print('is_movie:',        df['is_movie'].sum())
print('is_ongoing:',      df['is_ongoing'].sum())
print('is_completed:',    df['is_completed'].sum())

type_simplified: {'TV': 4515, 'ONA': 1727, 'Movie': 1623, 'OVA': 1353, 'Other': 924}
is_movie: 1623
is_ongoing: 169
is_completed: 9472


C:\Users\AQTAU\AppData\Local\Temp\ipykernel_18980\2655078106.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['is_completed'] = status_lower.str.contains(r'вышел|completed|завершён', regex=True).astype(int)
C:\Users\AQTAU\AppData\Local\Temp\ipykernel_18980\2655078106.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['is_ongoing'] = status_lower.str.contains(r'онго|ongoing|выходит|выпускается', regex=True).astype(int)
C:\Users\AQTAU\AppData\Local\Temp\ipykernel_18980\2655078106.py:10: PerformanceWarning: DataFrame is h

## Features 18–19: Source

In [10]:
ADAPTATION_SOURCES = r'манга|ранобэ|роман|light novel|manga|novel|игра|game|visual novel|веб-манга'

# Feature 18: is_adaptation
df['is_adaptation'] = (
    df['source'].fillna('').str.lower()
    .str.contains(ADAPTATION_SOURCES, regex=True)
    .astype(int)
)

# Feature 19: is_original_work
df['is_original_work'] = (
    df['source'].fillna('').str.lower()
    .str.contains(r'оригинальная идея|original', regex=True)
    .astype(int)
)

print('is_adaptation:',    df['is_adaptation'].value_counts().to_dict())
print('is_original_work:', df['is_original_work'].value_counts().to_dict())

is_adaptation: {1: 5876, 0: 4266}
is_original_work: {0: 7820, 1: 2322}


C:\Users\AQTAU\AppData\Local\Temp\ipykernel_18980\951535237.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['is_adaptation'] = (
C:\Users\AQTAU\AppData\Local\Temp\ipykernel_18980\951535237.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['is_original_work'] = (


## Features 20–23: Dubbing

In [11]:
# Feature 20: has_dubbing
df['has_dubbing'] = df['dubbing_list'].apply(
    lambda lst: int(len(lst) > 0 and not all(x.lower() == 'unknown' for x in lst))
)

# Feature 21: has_unknown_dubbing
df['has_unknown_dubbing'] = df['dubbing_list'].apply(
    lambda lst: int(any(x.lower() == 'unknown' for x in lst))
)

# Feature 22: dubbing_count
df['dubbing_count'] = df['dubbing_list'].apply(len)

# Feature 23: primary_dubbing
df['primary_dubbing'] = df['dubbing_list'].apply(lambda x: x[0] if x else 'Unknown')

print('has_dubbing:',         df['has_dubbing'].value_counts().to_dict())
print('has_unknown_dubbing:', df['has_unknown_dubbing'].value_counts().to_dict())

has_dubbing: {1: 8360, 0: 1782}
has_unknown_dubbing: {0: 8360, 1: 1782}


C:\Users\AQTAU\AppData\Local\Temp\ipykernel_18980\3849872681.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['has_dubbing'] = df['dubbing_list'].apply(
C:\Users\AQTAU\AppData\Local\Temp\ipykernel_18980\3849872681.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['has_unknown_dubbing'] = df['dubbing_list'].apply(
C:\Users\AQTAU\AppData\Local\Temp\ipykernel_18980\3849872681.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor perform

## Features 24–30: Studio and Director

In [12]:
# Feature 24: studio_count
df['studio_count'] = df['studio_list'].apply(len)

# Feature 25: primary_studio
df['primary_studio'] = df['studio_list'].apply(lambda x: x[0] if x else 'Unknown')

# Feature 26: studio_tier
TOP_STUDIOS = {
    'ufotable', 'mappa', 'wit studio', 'bones', 'kyoto animation',
    'madhouse', 'trigger', 'a-1 pictures', 'cloverworks', 'j.c.staff',
    'sunrise', 'toei animation', 'gainax', 'studio ghibli', 'p.a. works',
    'shaft', 'deen', 'production i.g', 'white fox', 'studio colorido',
    'tms entertainment',
}
MID_STUDIOS = {
    'silver link', 'feel', 'kinema citrus', 'seven arcs', 'doga kobo',
    'bridge', 'satelight', 'pine jam', 'lay-duce', 'studio bind',
    'c2c', 'connect', 'liden films', 'hs pictures studio',
}

def studio_tier(studio_str):
    if not studio_str or studio_str == 'Unknown':
        return 'Unknown'
    s = studio_str.lower()
    for top in TOP_STUDIOS:
        if top in s: return 'Top'
    for mid in MID_STUDIOS:
        if mid in s: return 'Mid'
    return 'Indie'

df['studio_tier'] = df['primary_studio'].apply(studio_tier)

# Features 27-28: studio_frequency, is_prolific_studio
studio_counts = df['primary_studio'].value_counts()
df['studio_frequency']   = df['primary_studio'].map(studio_counts).fillna(0).astype(int)
df['is_prolific_studio'] = (df['studio_frequency'] > 1).astype(int)

# Features 29-30: director_frequency, is_prolific_director
df['primary_director']     = df['director_list'].apply(lambda x: x[0] if x else 'Unknown')
director_counts            = df['primary_director'].value_counts()
df['director_frequency']   = df['primary_director'].map(director_counts).fillna(0).astype(int)
df['is_prolific_director'] = (df['director_frequency'] > 1).astype(int)

print('studio_tier:',          df['studio_tier'].value_counts().to_dict())
print('is_prolific_studio:',   df['is_prolific_studio'].sum())
print('is_prolific_director:', df['is_prolific_director'].sum())

C:\Users\AQTAU\AppData\Local\Temp\ipykernel_18980\3864879267.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['studio_count'] = df['studio_list'].apply(len)
C:\Users\AQTAU\AppData\Local\Temp\ipykernel_18980\3864879267.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['primary_studio'] = df['studio_list'].apply(lambda x: x[0] if x else 'Unknown')
C:\Users\AQTAU\AppData\Local\Temp\ipykernel_18980\3864879267.py:31: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` m

studio_tier: {'Indie': 5929, 'Top': 3227, 'Mid': 542, 'Unknown': 444}
is_prolific_studio: 9788
is_prolific_director: 9074


C:\Users\AQTAU\AppData\Local\Temp\ipykernel_18980\3864879267.py:41: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['director_frequency']   = df['primary_director'].map(director_counts).fillna(0).astype(int)
C:\Users\AQTAU\AppData\Local\Temp\ipykernel_18980\3864879267.py:42: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['is_prolific_director'] = (df['director_frequency'] > 1).astype(int)


## Feature 31: Hidden Gem

In [13]:
votes_30pct = df['site_votes_num'].quantile(0.30)

df['hidden_gem'] = (
    (df['site_rating_num'] >= 7.5) &
    (df['site_votes_num']  <= votes_30pct) &
    (df['site_votes_num']  > 0)
).astype(int)

print(f"hidden_gem: {df['hidden_gem'].sum()} anime")
print(df[df['hidden_gem'] == 1][['title_ru', 'site_rating_num', 'site_votes_num']].head(10))

hidden_gem: 77 anime
                                              title_ru  site_rating_num  \
221                        Сессионный оркестр принцесс             8.00   
230                                      Ты вкусняшка?             7.67   
292                     Югио! Тёмная сторона измерений             7.60   
554                    Камера, мотор! Фильм о монстрах             8.00   
619                                        Король ночи             7.67   
620                  Дело ведет юный детектив Киндаити             7.60   
628                                Музик Тигр в лесу 2             8.67   
691            Сильваниан Фэмилиз: Частица тайны Фрейи             7.67   
784                   Мелодия русалки: Пити Пити Пич 2             7.50   
912  Живая любовь! Клуб идолов старшей школы Нидзиг...             8.00   

     site_votes_num  
221              15  
230              15  
292              15  
554               2  
619               9  
620              15  

C:\Users\AQTAU\AppData\Local\Temp\ipykernel_18980\1476023023.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['hidden_gem'] = (


## Features 32–34: Log-scaled and Composite

In [14]:
# Feature 32: log_votes
df['log_votes'] = np.log1p(df['site_votes_num'].fillna(0))

# Feature 33: log_views
df['log_views'] = np.log1p(df['site_views_num'].fillna(0))

# Feature 34: composite_score
max_rating = df['site_rating_num'].max()
max_views  = df['site_views_num'].max()

df['composite_score'] = (
    0.6 * df['site_rating_num'].fillna(0) / (max_rating if max_rating else 1) +
    0.4 * df['site_views_num'].fillna(0)  / (max_views  if max_views  else 1)
).round(4)

print('composite_score — mean:', df['composite_score'].mean().round(4))

composite_score — mean: 0.4237


C:\Users\AQTAU\AppData\Local\Temp\ipykernel_18980\718246351.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['log_votes'] = np.log1p(df['site_votes_num'].fillna(0))
C:\Users\AQTAU\AppData\Local\Temp\ipykernel_18980\718246351.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['log_views'] = np.log1p(df['site_views_num'].fillna(0))
C:\Users\AQTAU\AppData\Local\Temp\ipykernel_18980\718246351.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which h

## Features 35–37: Title Metadata and Slug

In [15]:
# Feature 35: alt_names_count
df['alt_names_count'] = df['alt_names_list'].apply(len)

# Feature 36: title_ru_length
df['title_ru_length'] = df['title_ru'].str.len().fillna(0).astype(int)

# Feature 37: slug
df['slug'] = df['url'].str.rstrip('/').str.split('/').str[-1]

print(df[['title_ru', 'slug']].head(5))

C:\Users\AQTAU\AppData\Local\Temp\ipykernel_18980\1968090474.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['alt_names_count'] = df['alt_names_list'].apply(len)
C:\Users\AQTAU\AppData\Local\Temp\ipykernel_18980\1968090474.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['title_ru_length'] = df['title_ru'].str.len().fillna(0).astype(int)


                                            title_ru  \
0                   Моя история продолжается сегодня   
1  Детектив Конан: Цель — Когоро! Секретное рассл...   
2    Сердце дракона: Хроника исследования мира духов   
3                                    Госпожа Кэйфуку   
4               Ветер из Лояна: Таинственный мальчик   

                                    slug  
0  moya-istoriya-prodolzhaetsya-segodnya  
1                   detektiv-konan-ova-5  
2           dragon-heart-reikai-tanbouki  
3                       gospozha-keyfuku  
4   veter-iz-loyana-tainstvennyy-malchik  


C:\Users\AQTAU\AppData\Local\Temp\ipykernel_18980\1968090474.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['slug'] = df['url'].str.rstrip('/').str.split('/').str[-1]


## Summary

In [16]:
NEW_FEATURES = [
    'rating_tier', 'popularity_score', 'engagement_rate',
    'is_multi_genre', 'primary_genre',
    'decade', 'era', 'title_age', 'is_recent',
    'is_completed', 'is_ongoing', 'is_movie', 'type_simplified',
    'age_rating_ordinal', 'is_family_friendly',
    'is_adaptation', 'is_original_work',
    'has_dubbing', 'has_unknown_dubbing', 'dubbing_count', 'primary_dubbing',
    'studio_count', 'primary_studio', 'studio_tier',
    'studio_frequency', 'is_prolific_studio',
    'primary_director', 'director_frequency', 'is_prolific_director',
    'hidden_gem',
    'log_votes', 'log_views', 'composite_score',
    'alt_names_count', 'title_ru_length', 'slug',
] + genre_cols

print(f'Total new features: {len(NEW_FEATURES)}')
df[NEW_FEATURES].describe(include='all').T[['count', 'unique', 'top', 'mean', 'std', 'min', 'max']]

Total new features: 143


,count,unique,top,mean,std,min,max
rating_tier,10142,6,B,NaN,NaN,NaN,NaN
popularity_score,10142.0,<NA>,<NA>,22.199337,18.742486,0.0,100.0
engagement_rate,10142.0,<NA>,<NA>,29.19007,311.72511,0.0,8347.0
is_multi_genre,10142.0,NaN,NaN,0.835338,0.370893,0.0,1.0
primary_genre,10142,48,Сёнэн,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...
genre_киборги,10142.0,NaN,NaN,0.000493,0.022199,0.0,1.0
genre_силовые_костюмы,10142.0,NaN,NaN,0.000394,0.019857,0.0,1.0
genre_темные_эльфы,10142.0,NaN,NaN,0.000394,0.019857,0.0,1.0
genre_антивойна,10142.0,NaN,NaN,0.000197,0.014042,0.0,1.0


## Save the Final Dataset

In [17]:
KEEP_ORIGINAL = [
    'anime_id', 'title_ru', 'alt_names', 'status', 'type', 'year',
    'age_rating', 'genres', 'source', 'studio', 'director', 'dubbing',
    'site_rating', 'site_votes', 'site_views', 'url',
]
keep_orig = [c for c in KEEP_ORIGINAL if c in df.columns]

HELPER_COLS = [
    'alt_names_list', 'genres_list', 'studio_list', 'director_list', 'dubbing_list',
    'site_rating_num', 'site_votes_num', 'site_views_num', 'year_num',
]
df.drop(columns=[c for c in HELPER_COLS if c in df.columns], inplace=True)

final_cols = keep_orig + [f for f in NEW_FEATURES if f not in keep_orig and f in df.columns]
df_final   = df[final_cols].copy()

out_path = Path('../../data/processed/anime_features.csv')
out_path.parent.mkdir(parents=True, exist_ok=True)
df_final.to_csv(out_path, index=False, encoding='utf-8-sig')

print(f'Saved: {out_path}')
print(f'   Rows:    {len(df_final)}')
print(f'   Columns: {len(df_final.columns)}')
df_final.head(3)

Saved: ..\..\data\processed\anime_features.csv
   Rows:    10142
   Columns: 159


,anime_id,title_ru,alt_names,status,type,year,age_rating,genres,source,studio,...,genre_трансформеры,genre_искусственный_интеллект,genre_суккубы,genre_охотники_за_головами,genre_вестерн,genre_киборги,genre_силовые_костюмы,genre_темные_эльфы,genre_антивойна,genre_воры
0,10800,Моя история продолжается сегодня,"['Kyou, Watashi no Monogatari ga Hashirimasu. ...",вышел,ONA,2023,G,"['Повседневность ', ' Спорт']",Оригинальная идея,['Studio Colorido'],...,0,0,0,0,0,0,0,0,0,0
1,8258,Детектив Конан: Цель — Когоро! Секретное рассл...,"[""Detective Conan OVA 05: The Target is Kogoro...",вышел,OVA,2005,PG-13,"['Сёнэн ', ' Детектив ', ' Комедия ', ' Приклю...",Манга,['TMS Entertainment'],...,0,0,0,0,0,0,0,0,0,0
2,18742,Сердце дракона: Хроника исследования мира духов,['Dragon Heart: Adventures Beyond This World '...,вышел,Полнометражный фильм,2025,G,"['Фэнтези ', ' Драконы ', ' Сверхъестественное']",Оригинальная идея,['HS Pictures Studio'],...,0,0,0,0,0,0,0,0,0,0
